# Conditional GAN

Train on real training images containing pinhole or delamination, then generate the same five minority label combinations used by the VAE.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

sys.path.append(str(Path.cwd()))
from multilabel_utils import LABEL_COLUMNS, get_device, set_seed
from generative_models import ConditionalGenerator, ConditionalDiscriminator

SEED = 42
LATENT_DIM = 100
EPOCHS = 100
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "multilabel"
IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "archive" / "classification" / "images"
MODEL_DIR = PROJECT_ROOT / "models" / "multilabel"
OUTPUT_DIR = PROJECT_ROOT / "data" / "synthetic" / "multilabel_gan"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
train_df = train_df[(train_df["Delamination"] == 1) | (train_df["Pinhole"] == 1)].copy()
print("GAN training images:", len(train_df))

GAN training images: 483


In [2]:
gan_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

class GANDataset(Dataset):
    def __len__(self):
        return len(train_df)

    def __getitem__(self, index):
        row = train_df.iloc[index]
        image = Image.open(IMAGE_DIR / row["file_name"]).convert("RGB")
        condition = torch.tensor(
            row[LABEL_COLUMNS].to_numpy(dtype="float32"),
            dtype=torch.float32,
        )
        return gan_transform(image), condition

train_loader = DataLoader(
    GANDataset(), batch_size=16, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

In [3]:
generator = ConditionalGenerator(LATENT_DIM, len(LABEL_COLUMNS)).to(device)
discriminator = ConditionalDiscriminator(len(LABEL_COLUMNS)).to(device)

generator_optimizer = torch.optim.Adam(
    generator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999)
)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
    for real_images, conditions in train_loader:
        real_images = real_images.to(device)
        conditions = conditions.to(device)
        batch_size = real_images.size(0)
        real_targets = torch.full((batch_size,), 0.9, device=device)
        fake_targets = torch.zeros(batch_size, device=device)

        latent_vectors = torch.randn(batch_size, LATENT_DIM, device=device)
        fake_images = generator(latent_vectors, conditions)

        discriminator_optimizer.zero_grad()
        real_loss = criterion(
            discriminator(real_images, conditions), real_targets
        )
        fake_loss = criterion(
            discriminator(fake_images.detach(), conditions), fake_targets
        )
        discriminator_loss = real_loss + fake_loss
        discriminator_loss.backward()
        discriminator_optimizer.step()

        generator_optimizer.zero_grad()
        generator_loss = criterion(
            discriminator(fake_images, conditions), real_targets
        )
        generator_loss.backward()
        generator_optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"D loss: {discriminator_loss.item():.3f} | "
            f"G loss: {generator_loss.item():.3f}"
        )

generator_path = MODEL_DIR / "conditional_gan_generator.pth"
torch.save(generator.state_dict(), generator_path)

Epoch 10/100 | D loss: 3.059 | G loss: 0.708


Epoch 20/100 | D loss: 1.100 | G loss: 2.768


Epoch 30/100 | D loss: 0.372 | G loss: 3.438


Epoch 40/100 | D loss: 0.492 | G loss: 5.130


Epoch 50/100 | D loss: 1.746 | G loss: 0.345


Epoch 60/100 | D loss: 2.810 | G loss: 0.531


Epoch 70/100 | D loss: 2.073 | G loss: 1.490


Epoch 80/100 | D loss: 1.160 | G loss: 1.777


Epoch 90/100 | D loss: 2.643 | G loss: 1.062


Epoch 100/100 | D loss: 1.226 | G loss: 1.261


In [4]:
conditions_to_generate = {
    "delamination": [0, 1, 0],
    "pinhole": [0, 0, 1],
    "crack_delamination": [1, 1, 0],
    "crack_pinhole": [1, 0, 1],
    "all_three": [1, 1, 1],
}
samples_per_combination = 100
records = []
generator.eval()

with torch.no_grad():
    for condition_name, condition_values in conditions_to_generate.items():
        conditions = torch.tensor(
            condition_values, dtype=torch.float32, device=device
        ).repeat(samples_per_combination, 1)
        latent_vectors = torch.randn(
            samples_per_combination, LATENT_DIM, device=device
        )
        images = (generator(latent_vectors, conditions).cpu() + 1) / 2

        for index, image in enumerate(images):
            filename = f"gan_{condition_name}_{index:04d}.png"
            save_image(image, OUTPUT_DIR / filename)
            records.append({
                "file_name": filename,
                "image_path": str(Path("data") / "synthetic" / "multilabel_gan" / filename),
                "Surface_Crack": condition_values[0],
                "Delamination": condition_values[1],
                "Pinhole": condition_values[2],
                "is_synthetic": True,
            })

metadata = pd.DataFrame(records)
metadata.to_csv(OUTPUT_DIR / "metadata.csv", index=False)
print("Generated images:", len(metadata))

Generated images: 500


## Limitation

GAN loss does not measure defect correctness. Generated samples must be visually inspected and evaluated using the unchanged real test set.